# CO7093 — Group 12 Coursework Solution
**Dataset:** UCI Diabetes 130-US hospitals (`diabetic_data.csv`)

**Group 12 members (anonymous)**
| University ID |
|---------------|
| sa1213 |
| nny3 |
| gm420 |

**Notebook:** `group12_solution.ipynb` (per brief: `group{N}_solution.ipynb`)

**How to run (local)**
1. Place `diabetic_data.csv` in the same folder as this notebook (project root).
2. Install: `pip install -r requirements.txt` (includes `pyspark`, `imbalanced-learn`).
3. **PySpark** needs a **Java 11+** JDK on `PATH` / `JAVA_HOME` (common on lab machines).
4. Run all cells top to bottom.

**How to run (Google Colab)**  
Use **Runtime → Run all**. Run the **“Colab setup”** cell first: it installs packages, OpenJDK 11, downloads the UCI zip, and copies `diabetic_data.csv` into `/content`. If the download fails, upload `diabetic_data.csv` manually when prompted.

**Colab RAM:** Part 3 defaults to **18,000 stratified rows** on Colab (**float32** dummies + memory cleanup). Part 2 auto-switches to **3-fold CV**, **80 trees** RF, shorter LR iterations. Full data: High-RAM runtime + `import os; os.environ["PART3_MAX_ROWS"]="0"` before Part 3.

**Parts**
- **Part 1:** Cleaning, missing data, EDA plots (Pandas / Matplotlib / Seaborn).
- **Part 2:** Three sklearn classifiers, stratified CV, **random undersampling** on training data, metrics.
- **Part 3:** PySpark pipeline on full data, **K-Means**, elbow + silhouette, **local classifiers** vs global model.


### Google Colab — run the next cell once per session
On your laptop / Jupyter locally, that cell **does nothing** (it only detects Colab).

In [ ]:
# Colab setup: installs deps, Java for Spark, and fetches diabetic_data.csv into /content
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    import shutil
    import subprocess
    import sys
    import urllib.request
    import zipfile
    from pathlib import Path

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "imbalanced-learn>=0.12",
            "pyspark>=3.5",
            "pyarrow>=14",
        ]
    )
    subprocess.run(
        ["apt-get", "-qq", "-y", "install", "openjdk-11-jdk-headless"],
        check=False,
    )
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

    data_zip = (
        "https://archive.ics.uci.edu/static/public/296/"
        "diabetes+130-us+hospitals+for+years+1999-2008.zip"
    )
    zpath = Path("/content/diabetes_130.zip")
    dest_csv = Path("/content/diabetic_data.csv")
    try:
        urllib.request.urlretrieve(data_zip, zpath)
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall("/content")
        hits = list(Path("/content").rglob("diabetic_data.csv"))
        if hits:
            shutil.copy(hits[0], dest_csv)
            with open(dest_csv, encoding="utf-8", errors="replace") as f:
                nlines = sum(1 for _ in f) - 1
            print("Colab: saved", dest_csv.resolve(), f"({nlines:,} data rows)")
        else:
            print("Zip extracted but diabetic_data.csv not found; upload manually (next).")
    except Exception as exc:
        print("Auto-download failed:", exc)
        print("Upload diabetic_data.csv using the file picker:")
        from google.colab import files

        uploaded = files.upload()
        if "diabetic_data.csv" in uploaded:
            Path("/content/diabetic_data.csv").write_bytes(uploaded["diabetic_data.csv"])
            print("Saved /content/diabetic_data.csv")
    print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
else:
    print("Not on Google Colab — skipped (use local diabetic_data.csv + venv).")


## Part 1 — Data exploration and preprocessing

In [ ]:
# Configuration and imports
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
rng = np.random.RandomState(RANDOM_STATE)

ROOT = Path.cwd()
DATA_PATH = ROOT / "diabetic_data.csv"
if not DATA_PATH.exists():
    DATA_PATH = Path("diabetic_data.csv")

print("Data path:", DATA_PATH.resolve())


In [ ]:
def load_raw(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    # Standardise placeholders used in this release of the dataset
    df = df.replace({"?": np.nan, "None": np.nan, "": np.nan})
    return df


df = load_raw(DATA_PATH)
print("Shape (raw):", df.shape)
df.head()


In [ ]:
# Target: early readmission within 30 days (binary, per coursework framing)
# Positive = readmitted in < 30 days; negative = NO or >30 days
df["readmitted_binary"] = (df["readmitted"] == "<30").astype(int)
print(df["readmitted"].value_counts())
print("\nBinary (<30 vs not):", df["readmitted_binary"].value_counts(normalize=True).round(3))


In [ ]:
# Inspect missingness (before cleaning)
miss = df.isna().mean().sort_values(ascending=False)
print("Top missing % (columns):")
print((miss * 100).head(15).round(2))


In [ ]:
def clean_diabetes_df(frame: pd.DataFrame) -> pd.DataFrame:
    d = frame.copy()
    # Identifiers: keep patient_nbr for grouping; drop encounter_id from features later
    # Weight: overwhelmingly missing in this CSV — drop (justify in report)
    if "weight" in d.columns:
        d = d.drop(columns=["weight"])
    # Zero-variance medication flags from data dictionary
    for c in ["examide", "citoglipton"]:
        if c in d.columns:
            d = d.drop(columns=[c])
    # Very sparse admin text fields — drop to control dimensionality + noise
    for c in ["payer_code", "medical_specialty"]:
        if c in d.columns:
            d = d.drop(columns=[c])
    # Simplify diagnosis codes (high cardinality)
    for c in ["diag_1", "diag_2", "diag_3"]:
        if c in d.columns:
            d[c] = d[c].fillna("unknown").astype(str).str.strip().str[:4]
    # Fill remaining object columns with explicit missing bucket
    for c in d.select_dtypes(include=["object"]).columns:
        if c in ("readmitted",):
            continue
        d[c] = d[c].fillna("Missing")
    return d


df_clean = clean_diabetes_df(df)
print("Shape after cleaning drops/fills:", df_clean.shape)


In [ ]:
# Remove near-constant columns (no predictive information)
row_n = len(df_clean)
const_cols = []
for c in df_clean.columns:
    if c in ("readmitted", "readmitted_binary", "patient_nbr", "encounter_id"):
        continue
    nu = df_clean[c].nunique(dropna=False)
    if nu <= 1:
        const_cols.append(c)
    elif df_clean[c].value_counts(normalize=True, dropna=False).iloc[0] > 0.999:
        const_cols.append(c)

if const_cols:
    df_clean = df_clean.drop(columns=const_cols)
    print("Dropped near-constant:", const_cols)
else:
    print("No near-constant columns found.")


In [ ]:
# Feature / target matrices (patient-level split to reduce leakage)
ID_COL = "patient_nbr"
y = df_clean["readmitted_binary"].astype(int)
groups = df_clean[ID_COL]

feature_drop = ["encounter_id", ID_COL, "readmitted", "readmitted_binary"]
X = df_clean.drop(columns=[c for c in feature_drop if c in df_clean.columns])

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
X.head()


In [ ]:
# Train / test split — GroupShuffleSplit by patient
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

print("Train rows:", len(X_train), "Test rows:", len(X_test))
print("Train positive rate:", y_train.mean().round(4))
print("Test positive rate:", y_test.mean().round(4))


In [ ]:
# Outliers on key numeric utilisation features (winsorise at 99th percentile, train only)
def winsorise_train_test(
    X_tr: pd.DataFrame, X_te: pd.DataFrame, cols: list[str], q: float = 0.99
) -> tuple[pd.DataFrame, pd.DataFrame]:
    tr, te = X_tr.copy(), X_te.copy()
    for c in cols:
        if c not in tr.columns:
            continue
        cap = tr[c].quantile(q)
        tr[c] = tr[c].clip(upper=cap)
        te[c] = te[c].clip(upper=cap)
    return tr, te


util_cols = [
    c
    for c in [
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
    ]
    if c in X_train.columns
]
X_train, X_test = winsorise_train_test(X_train, X_test, util_cols)
print("Winsorised columns:", util_cols)


### Part 1 — Required visualisations

In [ ]:
# 1) Target distribution — donut (3-class) + horizontal count bars (binary)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
c3 = df_clean["readmitted"].value_counts()
pal3 = sns.color_palette("Spectral", n_colors=len(c3))
axes[0].pie(
    c3.values,
    labels=c3.index,
    autopct="%1.1f%%",
    colors=pal3,
    pctdistance=0.78,
    wedgeprops=dict(width=0.45, edgecolor="white"),
    startangle=90,
)
axes[0].set_title("Readmission label mix (donut)")
cb = df_clean["readmitted_binary"].value_counts().sort_index()
lbl = ["Not early (<30d)", "Early readmit (<30d)"]
axes[1].barh(lbl, cb.values, color=["#3a86ff", "#ff006e"], height=0.45, edgecolor="black", linewidth=0.6)
axes[1].invert_yaxis()
axes[1].set_xlabel("Encounters")
for i, v in enumerate(cb.values):
    axes[1].text(v + max(cb.values) * 0.02, i, f"{v:,}", va="center", fontsize=10)
axes[1].set_title("Binary target (horizontal bar)")
plt.tight_layout()
plt.show()


In [ ]:
# 2) Age vs readmission — point estimates with 95% CI (not a plain bar chart)
age_order = sorted(df_clean["age"].unique(), key=lambda s: int(s.strip("[]").split("-")[0]))
plt.figure(figsize=(10, 4.5))
sns.pointplot(
    data=df_clean,
    x="age",
    y="readmitted_binary",
    order=age_order,
    errorbar=("ci", 95),
    capsize=0.06,
    color="#1b998b",
    markers="D",
    linestyles="-",
    linewidth=1.8,
)
plt.ylabel("Mean P(early readmit)")
plt.xlabel("Age band")
plt.title("Age trend with bootstrap-style CI (Seaborn pointplot)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# 3) Medications vs readmission — letter-value (boxen) plot
plt.figure(figsize=(9, 4.5))
sns.boxenplot(
    data=df_clean,
    x="readmitted_binary",
    y="num_medications",
    hue="readmitted_binary",
    palette=["#5c4d7d", "#d1495b"],
    dodge=False,
    legend=False,
    k_depth="proportion",
)
plt.xlabel("Binary target (0 = not <30d, 1 = <30d)")
plt.ylabel("Number of medications")
plt.title("Distribution shape: medications by readmission (boxenplot)")
plt.tight_layout()
plt.show()


In [ ]:
# 4) Correlation structure — clustered heatmap on a numeric subset (reorders rows/cols)
sub_num = numeric_features[: min(14, len(numeric_features))]
corr_sub = X_train[sub_num].corr()
if len(sub_num) >= 2:
    grid = sns.clustermap(
        corr_sub,
        cmap="vlag",
        center=0,
        figsize=(9.5, 9.5),
        dendrogram_ratio=0.1,
        cbar_kws={"label": "Pearson r"},
        linewidths=0.4,
        linecolor="white",
    )
    grid.fig.suptitle("Hierarchically clustered correlations (train, numeric subset)", y=1.02, fontsize=12)
    plt.show()
else:
    plt.figure(figsize=(6, 4))
    sns.heatmap(corr_sub, cmap="vlag", center=0, annot=True)
    plt.title("Correlation (small numeric set)")
    plt.tight_layout()
    plt.show()
corr = X_train[numeric_features].corr()
stack = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
strong = stack.stack()
strong = strong[strong > 0.5]
print("Strong correlations (|r|>0.5, full numeric set):\n", strong.head(20))


In [ ]:
# 5) Extra EDA — violin: length of stay by binary outcome (training patients)
plt.figure(figsize=(8, 4.5))
sns.violinplot(
    data=df_clean.iloc[train_idx],
    x="readmitted_binary",
    y="time_in_hospital",
    palette=["#457b9d", "#e63946"],
    inner="quart",
    cut=0,
)
plt.xlabel("Binary target")
plt.ylabel("Time in hospital (days)")
plt.title("LOS distribution by early readmission (violin + quartiles)")
plt.tight_layout()
plt.show()


In [ ]:
# Persist cleaned table for PySpark / report reproducibility
CLEAN_PATH = ROOT / "data_clean_group12.parquet"
try:
    df_clean.to_parquet(CLEAN_PATH, index=False)
    print("Saved:", CLEAN_PATH)
except Exception as e:
    print("Parquet save skipped (install pyarrow if needed):", e)
